<a href="https://colab.research.google.com/github/JdogLloyd1/DeepLearning5888Project/blob/main/5888_PiczakProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Imports and configuration

%pip install -q librosa

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

SEED = 99
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ZIP_PATH = '/content/drive/MyDrive/5888_Project/Dataset/piczak_dataset.zip'
EXTRACT_DIR = '/content/piczak_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/5888_Project/results/Phase1_Baseline'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
        zip_file.extractall(EXTRACT_DIR)

CSV_PATH = None
AUDIO_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'esc50.csv' in files:
        CSV_PATH = os.path.join(root, 'esc50.csv')

    wav_files = [file_name for file_name in files if file_name.endswith('.wav')]
    if len(wav_files) > 0 and AUDIO_DIR is None:
        AUDIO_DIR = root

print('Dataset CSV:  ', CSV_PATH)
print('Audio folder: ', AUDIO_DIR)
print('Output folder:', OUTPUT_DIR)
print('Audio files:  ', len([file_name for file_name in os.listdir(AUDIO_DIR) if file_name.endswith('.wav')]))

Mounted at /content/drive
Dataset CSV:   /content/piczak_dataset/esc50.csv
Audio folder:  /content/piczak_dataset/audio/audio
Output folder: /content/drive/MyDrive/5888_Project/results/Phase1_Baseline
Audio files:   2000


In [ ]:
# Step 2: Spectrogram and model hyperparameters

VARIANT_CONFIGS = {
    "short": {
        "segment_frames": 41,
        "segment_hop_frames": 20,   # ~50% overlap
        "num_epochs": 300,
        "learning_rate": 0.002,
    },
    "long": {
        "segment_frames": 101,
        "segment_hop_frames": 10,   # ~90% overlap
        "num_epochs": 150,
        "learning_rate": 0.01,
    },
}

# Shared spectrogram setup
SAMPLE_RATE = 22050
N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 60

# Shared model setup
NUM_CLASSES = 50
CONV1_FILTERS = 80
CONV2_FILTERS = 80
FC_UNITS = 5000
DROPOUT_RATE = 0.5

# Shared training setup
BATCH_SIZE = 1000
MOMENTUM = 0.9
WEIGHT_DECAY = 0.001


In [ ]:
# Step 3: Load and explore the ESC-50 metadata

df = pd.read_csv(CSV_PATH)

display(df.head())

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [ ]:
# Step 4: Feature extraction — log-mel spectrogram + delta

def extract_features(audio_path, sr=SAMPLE_RATE, n_fft=N_FFT,
                     hop_length=HOP_LENGTH, n_mels=N_MELS):
    """Load one ESC-50 clip and extract 2-channel features:
       channel 0 = log-mel spectrogram
       channel 1 = delta
    """
    y, _ = librosa.load(audio_path, sr=sr)

    # normalize waveform
    y = librosa.util.normalize(y)

    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )

    log_mel = librosa.power_to_db(mel_spec)
    delta = librosa.feature.delta(log_mel)

    features = np.stack([log_mel, delta], axis=0).astype(np.float32)
    return features


# Quick test on one file
test_file = os.path.join(AUDIO_DIR, df.iloc[0]['filename'])
test_feat = extract_features(test_file)

print('Feature shape for one clip:', test_feat.shape)
print('  Channel 0 (log-mel): min={:.2f}, max={:.2f}'.format(
    test_feat[0].min(), test_feat[0].max()))
print('  Channel 1 (delta):   min={:.2f}, max={:.2f}'.format(
    test_feat[1].min(), test_feat[1].max()))

Feature shape for one clip: (2, 60, 216)
  Channel 0 (log-mel): min=-62.73, max=17.27
  Channel 1 (delta):   min=-8.24, max=12.72


In [ ]:
# Step 5: Segment extraction from full spectrograms

def extract_segments(features, segment_frames, segment_hop_frames):
    """Extract overlapping spectrogram segments and discard silent ones."""
    n_frames = features.shape[2]
    segments = []

    for start in range(0, n_frames - segment_frames + 1, segment_hop_frames):
        segment = features[:, :, start:start + segment_frames]

        segment_energy = np.mean(np.abs(segment[0]))
        if segment_energy > 1e-6:
            segments.append(segment.astype(np.float32))

    return segments


# Quick test on one clip for both variants
for variant_name, config in VARIANT_CONFIGS.items():
    test_segments = extract_segments(
        test_feat,
        segment_frames=config["segment_frames"],
        segment_hop_frames=config["segment_hop_frames"]
    )

    print(f"{variant_name} variant")
    print(f"  Full spectrogram frames: {test_feat.shape[2]}")
    print(f"  Number of segments:      {len(test_segments)}")
    if len(test_segments) > 0:
        print(f"  Segment shape:           {test_segments[0].shape}")
    print()

short variant
  Full spectrogram frames: 216
  Number of segments:      9
  Segment shape:           (2, 60, 41)

long variant
  Full spectrogram frames: 216
  Number of segments:      12
  Segment shape:           (2, 60, 101)



In [ ]:
# Step 6: Build segment samples and wrap them in a Dataset

def build_segment_samples(dataframe, audio_dir, segment_frames, segment_hop_frames):
    samples = []

    for idx in range(len(dataframe)):
        row = dataframe.iloc[idx]
        audio_path = os.path.join(audio_dir, row['filename'])
        label = int(row['target'])
        filename = row['filename']

        features = extract_features(audio_path)
        segments = extract_segments(
            features,
            segment_frames=segment_frames,
            segment_hop_frames=segment_hop_frames
        )

        for segment in segments:
            samples.append((segment, label, filename))

        if (idx + 1) % 200 == 0:
            print(f'  Processed {idx + 1}/{len(dataframe)} clips...')

    print(f'  Total segments: {len(samples)} from {len(dataframe)} clips')
    return samples


class ESC50Dataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        segment, label, filename = self.samples[idx]
        segment_tensor = torch.from_numpy(segment).float()
        return segment_tensor, label, filename

In [ ]:
# Step 7: Define the Piczak's 2015 CNN model

class PiczakCNN(nn.Module):
    def __init__(self, segment_frames, num_classes=NUM_CLASSES, dropout=DROPOUT_RATE):
        super().__init__()

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        # Convolutional layers
        self.conv1 = nn.Conv2d(
            in_channels=2,
            out_channels=CONV1_FILTERS,
            kernel_size=(57, 6),
            stride=1,
            padding=0
        )
        self.pool1 = nn.MaxPool2d(kernel_size=(4, 3), stride=(1, 3))

        self.conv2 = nn.Conv2d(
            in_channels=CONV1_FILTERS,
            out_channels=CONV2_FILTERS,
            kernel_size=(1, 3),
            stride=1,
            padding=0
        )
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        # Flattened size depends on segment length
        self.flat_size = self._get_flat_size(segment_frames)

        # Fully connected layers
        self.fc1 = nn.Linear(self.flat_size, FC_UNITS)
        self.fc2 = nn.Linear(FC_UNITS, FC_UNITS)
        self.out = nn.Linear(FC_UNITS, num_classes)

        self._init_weights()

    def _get_flat_size(self, segment_frames):
        with torch.no_grad():
            x = torch.zeros(1, 2, N_MELS, segment_frames)
            x = self.relu(self.conv1(x))
            x = self.dropout(x)
            x = self.pool1(x)

            x = self.relu(self.conv2(x))
            x = self.pool2(x)

            return x.view(1, -1).shape[1]

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.dropout(x)
        x = self.pool1(x)

        x = self.relu(self.conv2(x))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.out(x)

        return x


# Quick verification for both variants
for variant_name, config in VARIANT_CONFIGS.items():
    model_test = PiczakCNN(segment_frames=config["segment_frames"]).to(device)

    dummy = torch.randn(1, 2, N_MELS, config["segment_frames"]).to(device)
    dummy_out = model_test(dummy)

    print(variant_name)
    print(f'  Input shape:  {dummy.shape}')
    print(f'  Output shape: {dummy_out.shape}')

    total_params = sum(parameter.numel() for parameter in model_test.parameters())
    trainable_params = sum(
        parameter.numel() for parameter in model_test.parameters() if parameter.requires_grad
    )
    print(f'  Total parameters:     {total_params:,}')
    print(f'  Trainable parameters: {trainable_params:,}')
    print()

    del model_test, dummy, dummy_out

short
  Input shape:  torch.Size([1, 2, 60, 41])
  Output shape: torch.Size([1, 50])
  Total parameters:     26,534,130
  Trainable parameters: 26,534,130

long
  Input shape:  torch.Size([1, 2, 60, 101])
  Output shape: torch.Size([1, 50])
  Total parameters:     29,334,130
  Trainable parameters: 29,334,130



In [ ]:
# Step 8: Build fold-specific datasets and dataloaders

def make_fold_dataloaders(dataframe, audio_dir, test_fold, variant_name):
    config = VARIANT_CONFIGS[variant_name]

    train_df = dataframe[dataframe["fold"] != test_fold].reset_index(drop=True)
    test_df = dataframe[dataframe["fold"] == test_fold].reset_index(drop=True)

    print(f'Variant: {variant_name}')
    print(f'Test fold: {test_fold}')
    print(f'Train clips: {len(train_df)}')
    print(f'Test clips:  {len(test_df)}')

    train_samples = build_segment_samples(
        train_df,
        audio_dir,
        segment_frames=config["segment_frames"],
        segment_hop_frames=config["segment_hop_frames"]
    )

    test_samples = build_segment_samples(
        test_df,
        audio_dir,
        segment_frames=config["segment_frames"],
        segment_hop_frames=config["segment_hop_frames"]
    )

    train_dataset = ESC50Dataset(train_samples)
    test_dataset = ESC50Dataset(test_samples)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return train_df, test_df, train_samples, test_samples, train_loader, test_loader

In [ ]:
# Step 9: Set up training helper

def run_one_epoch(model, data_loader, loss_function, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for segments, labels, filenames in data_loader:
        segments = segments.to(device)
        labels = labels.to(device)

        if is_training:
            optimizer.zero_grad()
            logits = model(segments)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                logits = model(segments)
                loss = loss_function(logits, labels)

        predictions = torch.argmax(logits, dim=1)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_count += batch_size

    average_loss = total_loss / total_count
    average_accuracy = total_correct / total_count
    return average_loss, average_accuracy


def train_one_fold(dataframe, audio_dir, test_fold, variant_name):
    config = VARIANT_CONFIGS[variant_name]

    train_df, test_df, train_samples, test_samples, train_loader, test_loader = make_fold_dataloaders(
        dataframe,
        audio_dir,
        test_fold=test_fold,
        variant_name=variant_name
    )

    model = PiczakCNN(
        segment_frames=config["segment_frames"],
        num_classes=NUM_CLASSES,
        dropout=DROPOUT_RATE
    ).to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=config["learning_rate"],
        momentum=MOMENTUM,
        nesterov=True,
        weight_decay=WEIGHT_DECAY
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
    }

    for epoch_num in range(config["num_epochs"]):
        train_loss, train_acc = run_one_epoch(
            model,
            train_loader,
            loss_function,
            optimizer=optimizer
        )

        test_loss, test_acc = run_one_epoch(
            model,
            test_loader,
            loss_function,
            optimizer=None
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        if (epoch_num + 1) % 10 == 0 or epoch_num == 0:
            print(
                f'Epoch {epoch_num + 1:03d}/{config["num_epochs"]:03d} | '
                f'train loss {train_loss:.4f}, acc {train_acc:.4f} | '
                f'test loss {test_loss:.4f}, acc {test_acc:.4f}'
            )

    return model, history


In [ ]:
# Step 10: Set up plot curves

def save_training_curves(history, variant_name, fold_number, output_dir):
    epoch_list = np.arange(1, len(history["train_loss"]) + 1)

    fig, axis_list = plt.subplots(1, 2, figsize=(12, 5))

    # Loss
    axis_list[0].plot(epoch_list, history["train_loss"], label="Train")
    axis_list[0].plot(epoch_list, history["test_loss"], label="Test")
    axis_list[0].set_title(f"{variant_name} fold {fold_number}: Loss")
    axis_list[0].set_xlabel("Epoch")
    axis_list[0].set_ylabel("Cross-Entropy Loss")
    axis_list[0].grid(True)
    axis_list[0].legend()

    # Accuracy
    axis_list[1].plot(epoch_list, history["train_acc"], label="Train")
    axis_list[1].plot(epoch_list, history["test_acc"], label="Test")
    axis_list[1].set_title(f"{variant_name} fold {fold_number}: Accuracy")
    axis_list[1].set_xlabel("Epoch")
    axis_list[1].set_ylabel("Accuracy")
    axis_list[1].grid(True)
    axis_list[1].legend()

    fig.tight_layout()

    plot_path = os.path.join(output_dir, f"{variant_name}_fold_{fold_number}_curves.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)

    print(f"Training curves saved to: {plot_path}")

In [ ]:
# Step 11: Run all variants across all 5 folds

SAVE_PLOTS = True

all_results = []
history_records = []
all_histories = {}

for variant_name in VARIANT_CONFIGS:
    for test_fold in range(1, 6):
        print('\n' + '=' * 60)
        print(f'Running {variant_name} variant — fold {test_fold}')
        print('=' * 60)

        model, history = train_one_fold(
            df,
            AUDIO_DIR,
            test_fold=test_fold,
            variant_name=variant_name
        )

        all_histories[(variant_name, test_fold)] = history

        all_results.append({
            'variant': variant_name,
            'fold': test_fold,
            'final_train_loss': history['train_loss'][-1],
            'final_train_acc': history['train_acc'][-1],
            'final_test_loss': history['test_loss'][-1],
            'final_test_acc': history['test_acc'][-1],
        })

        if SAVE_PLOTS:
            save_training_curves(history, variant_name, test_fold, OUTPUT_DIR)

        for epoch_idx in range(len(history['train_loss'])):
            history_records.append({
                'variant': variant_name,
                'fold': test_fold,
                'epoch': epoch_idx + 1,
                'train_loss': history['train_loss'][epoch_idx],
                'train_acc': history['train_acc'][epoch_idx],
                'test_loss': history['test_loss'][epoch_idx],
                'test_acc': history['test_acc'][epoch_idx],
            })

print('\n' + '=' * 60)
print('ALL VARIANTS AND FOLDS COMPLETE')
print('=' * 60)


Running short variant — fold 1
Variant: short
Test fold: 1
Train clips: 1600
Test clips:  400
  Processed 200/1600 clips...
  Processed 400/1600 clips...
  Processed 600/1600 clips...
  Processed 800/1600 clips...
  Processed 1000/1600 clips...
  Processed 1200/1600 clips...
  Processed 1400/1600 clips...
  Processed 1600/1600 clips...
  Total segments: 14400 from 1600 clips
  Processed 200/400 clips...
  Processed 400/400 clips...
  Total segments: 3600 from 400 clips
Epoch 001/300 | train loss 4.4186, acc 0.0195 | test loss 3.9090, acc 0.0161
Epoch 010/300 | train loss 3.8594, acc 0.0397 | test loss 3.8724, acc 0.0464
Epoch 020/300 | train loss 3.6494, acc 0.0774 | test loss 3.7236, acc 0.0914
Epoch 030/300 | train loss 3.4927, acc 0.1015 | test loss 3.6095, acc 0.1206
Epoch 040/300 | train loss 3.3288, acc 0.1180 | test loss 3.4671, acc 0.0858
Epoch 050/300 | train loss 2.9966, acc 0.1730 | test loss 3.3262, acc 0.1300
Epoch 060/300 | train loss 2.7831, acc 0.2337 | test loss 3.133

In [ ]:
# Step 12: Summarize results

results_df = pd.DataFrame(all_results)

summary_df = (
    results_df
    .groupby('variant')[['final_test_acc']]
    .agg(['mean', 'std'])
)

print('Accuracy summary by variant:')
print(summary_df.round(4))

results_path = os.path.join(OUTPUT_DIR, 'baseline_results_by_fold.csv')
results_df.to_csv(results_path, index=False)

print(f'\nResults saved to: {results_path}')

Accuracy summary by variant:
        final_test_acc        
                  mean     std
variant                       
long            0.0200  0.0000
short           0.3661  0.0331

Results saved to: /content/drive/MyDrive/5888_Project/results/Phase1_Baseline/baseline_results_by_fold.csv


In [ ]:
# Step 13: Save to CSV

def save_history_csv(history_records, output_dir):
    history_df = pd.DataFrame(history_records)

    history_path = os.path.join(output_dir, 'baseline_epoch_history.csv')
    history_df.to_csv(history_path, index=False)

    return history_df

In [ ]:
# -------------------------- Model Testing ----------------------------------
# to use shared Google Drive ESC Project Deep Learning 5888 Data
# 1. create shortcut to shared drive in your own Drive
# 2. make sure the path is the same as save_dir
MODEL_CONFIG = {
    "model_name": "piczak_cnn",
    "variant_name": "baseline",
    "num_epochs": 30,
    "learning_rate": 1e-3,
    "batch_size": 32,
    "dropout": 0.5,
    "save_dir": "/content/drive/MyDrive/ESC Project Deep Learning 5888 Data"
}

In [ ]:
from torch.utils.data import DataLoader

def get_train_test_loaders(
    dataframe,
    audio_dir,
    test_fold,
    variant_name="short",
    batch_size=32,
    num_workers=2,
):
    """
    Returns standardized train/test loaders using ESC-50 folds.
    Minimal wrapper around your existing pipeline.
    """

    # --- 1. Split folds ---
    train_df = dataframe[dataframe["fold"] != test_fold].reset_index(drop=True)
    test_df  = dataframe[dataframe["fold"] == test_fold].reset_index(drop=True)

    # --- 2. Build segment indices (reuse your existing function) ---
    train_samples = build_segment_samples(
        train_df,
        audio_dir,
        segment_frames=VARIANT_CONFIGS[variant_name]["segment_frames"],
        hop_frames=VARIANT_CONFIGS[variant_name]["hop_frames"],
    )

    test_samples = build_segment_samples(
        test_df,
        audio_dir,
        segment_frames=VARIANT_CONFIGS[variant_name]["segment_frames"],
        hop_frames=VARIANT_CONFIGS[variant_name]["hop_frames"],
    )

    # --- 3. Create datasets (reuse your class) ---
    train_dataset = ESC50Dataset(train_samples)
    test_dataset  = ESC50Dataset(test_samples)

    # --- 4. Create loaders ---
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, test_loader

In [ ]:
#copy this method and modify the contents for your training model
#change method name to your training name
#each person must define the model to pass in
#train_loader, test_loader already defined
def run_model(model, train_loader, test_loader, config):
    # use your existing training loop here
      # collect:
      # - train_loss
      # - train_acc
      # - val_acc
      # - test_acc
      # - training time
    # return them in a dictionary
    pass

In [ ]:
def save_results(history, results, config):
    import os, json
    from pathlib import Path

    save_path = Path(config["save_dir"]) / config["model_name"] / config["variant_name"]
    save_path.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(history).to_csv(save_path / "history.csv", index=False)
    pd.DataFrame([results]).to_csv(save_path / "results.csv", index=False)

    with open(save_path / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    plt.savefig(save_path / "curves.png", dpi=200, bbox_inches="tight")

In [ ]:
#for example
#train_loader, test_loader = get_train_test_loaders(
#    dataframe=df,
#    audio_dir=AUDIO_DIR,
#    test_fold=0,
#    variant_name="short",
#    batch_size=MODEL_CONFIG["batch_size"],
#)
#AST_model = ASTModel(num_classes=NUM_CLASSES)
#history, results = run_model(AST_model, train_loader, test_loader, MODEL_CONFIG)
#save_results(history, results, MODEL_CONFIG)